# 13 — Kiểm tra kỹ thuật và chuẩn bị đánh giá gợi ý

**Câu hỏi:** điều gì đã được kiểm chứng bằng code, và điều gì cần GVHD/người học đánh giá?

Notebook này không tạo điểm chấm giả. Phần một chạy kiểm thử tự động; phần hai tạo phiếu trống cho người đánh giá. Chạy 11 và 12 trước.

In [1]:
from pathlib import Path
import json
import os
import sys
import pandas as pd
from IPython.display import display

current = Path.cwd()
ROOT = next(parent for parent in [current, *current.parents] if (parent / 'apps/backend/app').is_dir())
sys.path.insert(0, str(ROOT / 'apps'))
sys.path.insert(0, str(ROOT / 'apps/backend'))
RESEARCH = Path(os.environ.get('CLHT_RESEARCH_DATA', str(ROOT.parent / 'CLHT_GB_GITHUB')))
OUTPUT = ROOT / 'apps/research_outputs'
FIGURES = OUTPUT / 'hinh'
OUTPUT.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
print('Mốc phục vụ app: 75%. Không đổi model và không dùng nhãn cuối kỳ để sinh gợi ý.')


Mốc phục vụ app: 75%. Không đổi model và không dùng nhãn cuối kỳ để sinh gợi ý.


## 1. Kiểm thử API và các tình huống biên

Bao gồm bài nộp sớm, thiếu hạn, bảo lưu, miễn bài, cutoff sai, sự kiện sau cutoff, cửa sổ 7/30 ngày, từ chối khi thiếu dữ liệu và model thật. Kiểm tra pass nghĩa là các điều kiện kỹ thuật được kiểm tra chạy đúng, không chứng minh tác động học tập.

In [2]:
import subprocess
completed = subprocess.run([sys.executable, '-m', 'pytest', 'tests', '-q'],
    cwd=ROOT / 'apps/backend', capture_output=True, text=True, encoding='utf-8')
print(completed.stdout)
if completed.returncode:
    print(completed.stderr)
assert completed.returncode == 0, 'Cần sửa lỗi trước khi nghiệm thu.'
(OUTPUT / 'kiem_thu_api.txt').write_text(completed.stdout, encoding='utf-8')

...................................                                      [100%]
============================== warnings summary ===============================
.venv\Lib\site-packages\starlette\testclient.py:45
  C:\Users\nguye\OneDrive - HUFI\Nam_4\KL_CN\CLHT_GB_APP_WORKTREE\apps\backend\.venv\Lib\site-packages\starlette\testclient.py:45: DeprecationWarning: The anyio.abc.BlockingPortal alias is deprecated, use anyio.from_thread.BlockingPortal instead.
    _PortalFactoryType = Callable[[], AbstractContextManager[anyio.abc.BlockingPortal]]

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
35 passed, 1 warning in 4.69s



649

## 2. Xác nhận điểm nguy cơ không làm mất căn cứ

Tình huống dưới chỉ mock xác suất 1% để kiểm tra logic tích hợp, không phải xác suất đo trên dữ liệu. Phải còn G01 nếu có bài đến hạn chưa có bản ghi.

In [3]:
from unittest.mock import patch
from fastapi.testclient import TestClient
from app.main import app
from app.services.support import demo_cases
case = next(item for item in demo_cases() if item['id'] == 'bai-den-han')
with patch('app.services.support.predict', return_value={'probability_risk': .01}):
    response = TestClient(app).post('/api/v1/support', json=case['payload'])
assert response.status_code == 200
assert 'G01' in [item['rule_id'] for item in response.json()['suggestions']]
print('PASS: mô phỏng nguy cơ 1% không chặn gợi ý G01 có căn cứ. Không dùng xác suất mock cho báo cáo kết quả model.')

PASS: mô phỏng nguy cơ 1% không chặn gợi ý G01 có căn cứ. Không dùng xác suất mock cho báo cáo kết quả model.


## 3. Phiếu chấm độ phù hợp — chưa có người chấm

So sánh lời khuyên chung và lời khuyên theo hồ sơ trên cùng dữ liệu tại cutoff. Ẩn phương pháp, đảo thứ tự và không đưa nhãn cuối kỳ. Dự kiến chấm 1–5 cho phù hợp, căn cứ, khả thi, dễ hiểu. Cỡ mẫu và tiêu chí cần thống nhất với cô. Các ô điểm dưới đây cố ý để trống.

In [4]:
from app.services.support import demo_cases
columns = ['ma_ho_so', 'phuong_an_an_danh', 'nguoi_cham', 'ngay_cham',
           'phu_hop_1_5', 'co_can_cu_1_5', 'kha_thi_1_5', 'de_hieu_1_5', 'nhan_xet']
review = pd.DataFrame([{column: None for column in columns} for _ in range(2 * len(demo_cases()))])
review['ma_ho_so'] = [case['id'] for case in demo_cases() for _ in range(2)]
assert review[['phu_hop_1_5', 'co_can_cu_1_5', 'kha_thi_1_5', 'de_hieu_1_5']].isna().all().all()
review.to_csv(OUTPUT / 'phieu_danh_gia_chua_cham.csv', index=False, encoding='utf-8-sig')
display(review.head(6))
print('Trạng thái: chưa đánh giá bởi người dùng/giảng viên; chưa có điểm trung bình để báo cáo.')

,ma_ho_so,phuong_an_an_danh,nguoi_cham,ngay_cham,phu_hop_1_5,co_can_cu_1_5,kha_thi_1_5,de_hieu_1_5,nhan_xet
0,bai-den-han,None,None,None,None,None,None,None,None
1,bai-den-han,None,None,None,None,None,None,None,None
2,nop-tre,None,None,None,None,None,None,None,None
3,nop-tre,None,None,None,None,None,None,None,None
4,gian-doan-7,None,None,None,None,None,None,None,None
5,gian-doan-7,None,None,None,None,None,None,None,None


Trạng thái: chưa đánh giá bởi người dùng/giảng viên; chưa có điểm trung bình để báo cáo.


## 4. Có thể nói gì với cô?

Đã có luồng dữ liệu → model thật → gợi ý có căn cứ → app, cùng kiểm thử và notebook tái lập. Chưa có bằng chứng tăng điểm; chưa tối ưu trực tiếp chiến lược bằng HCO–VNLS; chưa có đánh giá công bằng và kỳ học mới độc lập. Đây là giới hạn cần trình bày, không được thay bằng chỉ số dự báo.

**Bài tập:** giải thích vì sao test API pass không đồng nghĩa chiến lược giúp tăng điểm. Gợi ý trả lời: test xác nhận code làm đúng quy tắc; tác dụng của quy tắc lên người học là câu hỏi thực nghiệm khác.